# Day 27 Tutorial：论文到字段的可追溯映射

> **所有表格都是文献字段映射示例/待确认模板。** 它们不是论文原始数据，不是本组实验数据，禁止用于模型训练。

## Goal

把现有任务卡中的三篇论文整理成输入—条件—目标证据表，再生成通用/体系专用候选字段与不可拼表检查清单。

## Setup

本 Notebook 不联网、不抓取论文数值、不训练模型。DOI 仅作来源标识；数据许可与原始数据可得性仍需单独核查。

In [1]:
import json
import pandas as pd
from IPython.display import display

ARTIFACT_STATUS = "literature_mapping_example_not_training_data"
print("Artifact status:", ARTIFACT_STATUS)

Artifact status: literature_mapping_example_not_training_data


## Steps

### 1. 建立文献字段映射示例

内容来自本日已列出的文献摘要，只表达字段类别，不包含性能数值。

In [2]:
paper_mapping = pd.DataFrame([
    {
        "paper_id": "A",
        "doi": "10.1080/14686996.2019.1673670",
        "system": "epoxy / polyetheramine",
        "formulation_inputs": "resin MW; hardener MW; amine/epoxy ratio",
        "process_or_joint": "curing temperature; controlled lap joint",
        "target_category": "joint strength under stated lap-shear context",
        "raw_data_obtained": False,
        "status": ARTIFACT_STATUS,
    },
    {
        "paper_id": "B",
        "doi": "10.3390/nano11040872",
        "system": "multi-component structural epoxy",
        "formulation_inputs": "amounts of eight reported component classes",
        "process_or_joint": "reported preparation and test conditions",
        "target_category": "lap-shear and impact-peel tasks kept separate",
        "raw_data_obtained": False,
        "status": ARTIFACT_STATUS,
    },
    {
        "paper_id": "C",
        "doi": "10.1007/s00226-019-01144-6",
        "system": "1C polyurethane / beech wood",
        "formulation_inputs": "prepolymer composition/structure parameters",
        "process_or_joint": "wood substrate; pretreatment/environment",
        "target_category": "tensile-shear strength in stated context",
        "raw_data_obtained": False,
        "status": ARTIFACT_STATUS,
    },
])
display(paper_mapping)

,paper_id,doi,system,formulation_inputs,process_or_joint,target_category,raw_data_obtained,status
0,A,10.1080/14686996.2019.1673670,epoxy / polyetheramine,resin MW; hardener MW; amine/epoxy ratio,curing temperature; controlled lap joint,joint strength under stated lap-shear context,False,literature_mapping_example_not_training_data
1,B,10.3390/nano11040872,multi-component structural epoxy,amounts of eight reported component classes,reported preparation and test conditions,lap-shear and impact-peel tasks kept separate,False,literature_mapping_example_not_training_data
2,C,10.1007/s00226-019-01144-6,1C polyurethane / beech wood,prepolymer composition/structure parameters,wood substrate; pretreatment/environment,tensile-shear strength in stated context,False,literature_mapping_example_not_training_data


### 2. 把字段候选标记为通用、专用或待确认

In [3]:
field_candidates = pd.DataFrame([
    ("sample_id", "identity", "common_candidate", "待化学组确认行定义"),
    ("adhesive_family", "scope", "common_candidate", "待确认首个体系"),
    ("component_identity", "formulation", "common_candidate", "一组分一记录或宽表待定"),
    ("amount", "formulation", "common_candidate", "需与 amount_basis 配对"),
    ("curing_temperature", "process", "common_candidate", "单位待确认"),
    ("substrate", "joint", "common_candidate", "受测试体系影响"),
    ("test_standard", "target_context", "common_candidate", "必须与标签一起存"),
    ("replicate_count", "quality", "common_candidate", "重复关系待确认"),
    ("epoxy_equivalent_weight", "formulation", "system_specific", "仅环氧相关时启用"),
    ("wood_moisture_content", "joint", "system_specific", "仅木材体系相关时启用"),
    ("may_use_for_modeling", "permission", "pending_chemistry", "默认不得假定为是"),
    ("may_upload_to_github", "permission", "pending_chemistry", "未授权时默认 false"),
], columns=["field", "layer", "scope_status", "review_note"])
display(field_candidates)

,field,layer,scope_status,review_note
0,sample_id,identity,common_candidate,待化学组确认行定义
1,adhesive_family,scope,common_candidate,待确认首个体系
2,component_identity,formulation,common_candidate,一组分一记录或宽表待定
3,amount,formulation,common_candidate,需与 amount_basis 配对
4,curing_temperature,process,common_candidate,单位待确认
5,substrate,joint,common_candidate,受测试体系影响
6,test_standard,target_context,common_candidate,必须与标签一起存
7,replicate_count,quality,common_candidate,重复关系待确认
8,epoxy_equivalent_weight,formulation,system_specific,仅环氧相关时启用
9,wood_moisture_content,joint,system_specific,仅木材体系相关时启用


### 3. 生成安全 draft schema

所有关键值显著标注“待确认”，公开上传默认 `false`。

In [4]:
draft_schema = {
    "schema_version": "draft-v1.0",
    "status": "example_template_not_data",
    "research_scope": {
        "adhesive_family": "待化学组确认",
        "row_definition": "待确认：配方/试样/重复",
    },
    "target": {
        "property_name": "待确认",
        "unit": "待确认",
        "test_standard": "待确认",
    },
    "permissions": {
        "may_use_for_modeling": "待确认",
        "may_upload_to_github": False,
    },
    "paper_evidence": paper_mapping["doi"].tolist(),
}
print(json.dumps(draft_schema, ensure_ascii=False, indent=2))

{
  "schema_version": "draft-v1.0",
  "status": "example_template_not_data",
  "research_scope": {
    "adhesive_family": "待化学组确认",
    "row_definition": "待确认：配方/试样/重复"
  },
  "target": {
    "property_name": "待确认",
    "unit": "待确认",
    "test_standard": "待确认"
  },
  "permissions": {
    "may_use_for_modeling": "待确认",
    "may_upload_to_github": false
  },
  "paper_evidence": [
    "10.1080/14686996.2019.1673670",
    "10.3390/nano11040872",
    "10.1007/s00226-019-01144-6"
  ]
}


## Checks

确认没有性能数值列、没有宣称获得论文原始数据、状态标签完整。

In [5]:
forbidden_columns = {"target_value", "strength_value", "measured_value"}
assert forbidden_columns.isdisjoint(set(paper_mapping.columns))
assert not paper_mapping["raw_data_obtained"].any()
assert paper_mapping["status"].eq(ARTIFACT_STATUS).all()
assert field_candidates["review_note"].notna().all()
assert draft_schema["permissions"]["may_upload_to_github"] is False
print("Checks passed: mapping/template only; zero model training rows.")

Checks passed: mapping/template only; zero model training rows.


## Next Steps

让导师与化学组确认体系、行定义、首要目标、单位、测试标准、重复/批次和权限，再冻结 v1.0。不要把三篇来源中的“强度”数值纵向拼接。